In [1]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display

import pandas as pd
import numpy as np

from google import genai
from google.genai import types

In [3]:
from multiprocessing import Value


load_dotenv()

api_key = os.getenv('GEMINI_API_KEY')
if not api_key:
    raise ValueError('api key not loaded')

client = genai.Client(api_key=api_key)

In [10]:
model = 'gemini-3.5-flash'

In [12]:
response = client.models.generate_content(
    model=model,
    contents='explain what machine learning is in three sentences'
)

print(response.text)

Machine learning is a branch of artificial intelligence that enables computers to learn from data and make decisions without being explicitly programmed. By analyzing patterns within large datasets, machine learning models can recognize complex trends and accurately predict future outcomes. As these systems process more information over time, their accuracy and performance automatically improve without needing human intervention.


In [13]:
APP_INSTRUCTIONS = """
You are StudyMate, an educational assistant.

Your purpose is to help students understand and revise
their lecture materials.

Follow these principles:
- Be accurate and educational.
- Explain concepts clearly and logically.
- Prefer simple language without sacrificing accuracy.
- Use the provided lecture notes as your primary source.
- Do not invent information that is not supported by the notes.
"""

def generate_response(prompt):
    response = client.models.generate_content(
        model=model,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=APP_INSTRUCTIONS
        )
    )

    return response.text

In [14]:
def summarize_notes(notes):
    prompt = f"""
    Summarize the following lecture notes.

    Organize your response into these sections:

    1. Main idea
    2. Important concepts
    3. Important formulas
    4. Relationships between concepts
    5. Common misconceptions

    Use only the information provided in the notes.
    Explain the concepts clearly.

    LECTURE NOTES:
    {notes}
    """

    return generate_response(prompt)

In [15]:
notes = """
Logistic Regression

Logistic Regression is a supervised learning algorithm used
primarily for classification problems.

It calculates a linear combination of input features:

z = w1*x1 + w2*x2 + ... + wn*xn + b

The result is passed through the sigmoid function:

sigmoid(z) = 1 / (1 + exp(-z))

The sigmoid function maps any real number into the interval
between 0 and 1. This output can be interpreted as the
estimated probability of the positive class.

The model learns its parameters using Maximum Likelihood
Estimation (MLE).

The log-odds of the positive class are linear in the input
features.

Exponentiating a coefficient gives its odds ratio.
An odds ratio greater than 1 indicates increased odds,
while an odds ratio below 1 indicates decreased odds,
holding other features constant.

Despite its name, Logistic Regression is primarily used
for classification, not predicting continuous numerical values.
"""

summary = summarize_notes(notes)
display(Markdown(summary))

Hello! I am StudyMate, your educational assistant. Here is a clear, structured summary of your lecture notes on **Logistic Regression**.

---

### 1. Main idea
Logistic Regression is a supervised learning algorithm primarily used for classification tasks. It calculates a linear combination of input features and transforms it into a probability value between 0 and 1 to estimate the likelihood of a positive class.

---

### 2. Important concepts
* **Supervised Classification:** Logistic Regression is designed to categorize data into classes rather than predict continuous numerical values.
* **Sigmoid Function:** A mathematical function that converts any real-valued number into a value between 0 and 1, which represents the estimated probability of the positive class.
* **Maximum Likelihood Estimation (MLE):** The optimization method used by the model to learn its parameters (weights and bias).
* **Log-Odds:** The log-odds of belonging to the positive class have a linear relationship with the input features.
* **Odds Ratio:** Calculated by exponentiating a feature's coefficient ($w$). 
  * An odds ratio **greater than 1** indicates increased odds of the positive class.
  * An odds ratio **below 1** indicates decreased odds, assuming all other features remain constant.

---

### 3. Important formulas

* **Linear combination of features ($z$):**
  $$z = w_1 x_1 + w_2 x_2 + \dots + w_n x_n + b$$

* **Sigmoid function:**
  $$\text{sigmoid}(z) = \frac{1}{1 + \exp(-z)}$$

---

### 4. Relationships between concepts
* **Linear Combination $\rightarrow$ Sigmoid $\rightarrow$ Probability:** The input features ($x_i$) are multiplied by weights ($w_i$) and added to a bias ($b$) to produce $z$. This result ($z$) is then passed into the sigmoid function to map it to a probability between 0 and 1.
* **Input Features and Log-Odds:** The log-odds of the positive class depend linearly on the input features.
* **Coefficients and Odds Ratios:** Exponentiating a feature's learned coefficient directly yields its odds ratio, showing how changes in that feature affect the odds of the positive class while holding other features constant.
* **MLE and Parameters:** Maximum Likelihood Estimation (MLE) is the process used to find the optimal values for the parameters ($w$ and $b$).

---

### 5. Common misconceptions
* **Misconception:** *Logistic Regression is used to predict continuous numerical values.*
  * **Fact:** Despite containing the word "Regression" in its name, Logistic Regression is primarily used for **classification** problems, not for predicting continuous values.

In [19]:
FLASHCARD_SCHEMA = {
    "type": "OBJECT",
    "properties": {
        "flashcards": {
            "type": "ARRAY",
            "items": {
                "type": "OBJECT",
                "properties": {
                    "question": {"type": "STRING"},
                    "answer": {"type": "STRING"}
                },
                "required": ["question", "answer"]
            }
        }
    },
    "required": ["flashcards"]
}

QUIZ_SCHEMA = {
    "type": "OBJECT",
    "properties": {
        "questions": {
            "type": "ARRAY",
            "items": {
                "type": "OBJECT",
                "properties": {
                    "question": {"type": "STRING"},
                    "options": {
                        "type": "ARRAY",
                        "items": {"type": "STRING"}
                    },
                    "answer_index": {"type": "INTEGER"},
                    "explanation": {"type": "STRING"}
                },
                "required": [
                    "question",
                    "options",
                    "answer_index",
                    "explanation"
                ]
            }
        }
    },
    "required": ["questions"]
}

In [20]:
def generate_flashcards(notes, number=8):
    prompt = f"""
    Create {number} flashcards based on the provided lecture notes.

    Requirements:
    - Focus on important concepts.
    - Test conceptual understanding, not just memorization.
    - Keep questions clear and answers concise.
    - Do not introduce information absent from the notes.

    LECTURE NOTES:
    {notes}
    """

    response = client.models.generate_content(
        model=model,
        contents=prompt,
        config = types.GenerateContentConfig(
            system_instruction=APP_INSTRUCTIONS,
            response_mime_type='application/json',
            response_json_schema=FLASHCARD_SCHEMA
        )
    )

    return json.loads(response.text)

In [21]:
flashcards = generate_flashcards(notes, 8)
flashcards_df = pd.DataFrame(flashcards['flashcards'])

display(flashcards_df)

,question,answer
0,What type of algorithm is Logistic Regression ...,Logistic Regression is a supervised learning a...
1,What is the formula for the linear combination...,The linear combination is calculated as z = w1...
2,What function maps the linear output z into a ...,"The sigmoid function, defined as sigmoid(z) = ..."
3,How is the output of the sigmoid function inte...,The output is interpreted as the estimated pro...
4,By what method does Logistic Regression learn ...,Logistic Regression learns its parameters usin...
5,What is the mathematical relationship between ...,The log-odds of the positive class are linear ...
6,How is an odds ratio calculated from a model c...,An odds ratio is calculated by exponentiating ...
7,Why can the name 'Logistic Regression' be misl...,"Despite containing 'Regression' in its name, i..."


In [22]:
def generate_quiz(notes, number=4):
    prompt = f"""
    Create {number} multiple-choice questions based on
    the provided lecture notes.

    Requirements:
    - Four options per question.
    - Exactly one correct answer.
    - Test conceptual understanding.
    - Include an explanation of the correct answer.
    - answer_index must be an integer from 0 to 3.

    LECTURE NOTES:
    {notes}
    """

    response = client.models.generate_content(
        model=model,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=APP_INSTRUCTIONS,
            response_mime_type='application/json',
            response_json_schema=QUIZ_SCHEMA
        )
    )

    return json.loads(response.text)

In [23]:
quiz = generate_quiz(notes, 5)
quiz_df = pd.DataFrame(quiz['questions'])

display(quiz_df)

,question,options,answer_index,explanation
0,"Despite having 'Regression' in its name, what ...","[Predicting continuous numerical values, Class...",1,"According to the lecture notes, Logistic Regre..."
1,What is the main function of the sigmoid funct...,[To map any real number into the interval betw...,0,The sigmoid function maps any real number into...
2,Which method does Logistic Regression use to l...,"[Ordinary Least Squares, Gradient Boosted Tree...",2,The lecture notes state that the model learns ...
3,What is the relationship between the log-odds ...,"[They are quadratic, They are linear, They are...",1,The notes specify that the log-odds of the pos...
4,What does an exponentiating feature coefficien...,"[Decreased odds of the positive class, Increas...",1,Exponentiating a coefficient gives its odds ra...


In [24]:
def run_quiz(quiz):

    score = 0
    questions = quiz["questions"]

    for i, question in enumerate(questions, start=1):

        print(f"\nQuestion {i}: {question['question']}")

        for j, option in enumerate(question["options"], start=1):
            print(f"{j}. {option}")

        while True:
            try:
                answer = int(input("Your answer (1-4): "))

                if answer in range(1, 5):
                    break

                print("Please enter a number between 1 and 4.")

            except ValueError:
                print("Please enter a valid integer.")

        if answer - 1 == question["answer_index"]:
            print("\nCorrect!")
            score += 1

        else:
            correct = question["options"][question["answer_index"]]
            print(f"\nIncorrect. Correct answer: {correct}")

        print(f"Explanation: {question['explanation']}")

    print(f"\nFinal score: {score}/{len(questions)}")

    return score

In [ ]:
from textwrap import indent


flashcards_df.to_json('flashcards.json')
quiz_df.to_json('quiz.json', indent=4)